Step 1 - Load everything needed for inference

weights=None — this time we build the architecture without downloading ImageNet weights, since we're about to load our own trained weights instead — no point loading two sets of weights

Same classifier modification as training (must match exactly, or the saved weights won't load correctly)

map_location=device — ensures the saved weights load correctly onto whichever device (GPU/CPU) is available, avoiding a common error when loading GPU-trained weights on a machine without GPU

Same transform pipeline as training — critical: inference must preprocess images identically to how training data was preprocessed, or the model sees different-looking input than it learned on

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Rebuild the same architecture we trained
model = models.efficientnet_b0(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_best.pth", map_location=device))
model = model.to(device)
model.eval()

class_names = ['fake', 'real']  # matches train_data.classes order from training

IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Model loaded and ready for inference.")

Model loaded and ready for inference.


Step 2- Prediction function (with face-crop preprocessing)

Applies the exact same face-crop logic used during preprocessing — consistency between training and inference preprocessing is essential, mismatches here are a very common source of poor real-world performance

unsqueeze(0) — adds a "batch dimension" of size 1, since the model expects batches of images, not a single raw image
Returns a clean dictionary with prediction, confidence, and both class probabilities — exactly what we'll display in the Gradio UI next

In [2]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def predict_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return {"error": "Could not read image"}
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    
    if len(faces) > 0:
        faces = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces[0]
        face_crop = img[y:y+h, x:x+w]
    else:
        face_crop = img  # fallback to full image, same logic as preprocessing
    
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(face_rgb)
    
    input_tensor = transform(pil_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
        pred_idx = torch.argmax(probs).item()
        confidence = probs[pred_idx].item()
    
    return {
        "prediction": class_names[pred_idx],
        "confidence": round(confidence * 100, 2),
        "fake_probability": round(probs[0].item() * 100, 2),
        "real_probability": round(probs[1].item() * 100, 2)
    }

Step 3 - Quick test

In [3]:
import os
# Test on one image from your test set
test_image_path = "D:/Deepfake-detection/datasets/test/real/" + os.listdir("D:/Deepfake-detection/datasets/test/real")[0]
result = predict_image(test_image_path)
print(result)

{'prediction': 'real', 'confidence': 100.0, 'fake_probability': 0.0, 'real_probability': 100.0}


Step 4 - install the Hugging Face Hub client (if not already active)

token already taken

In [4]:
import huggingface_hub
print(huggingface_hub.__version__)

1.24.0


Step 5 - Set up the client with your token

In [1]:
from huggingface_hub import InferenceClient

from dotenv import load_dotenv
import os

load_dotenv("D:/Deepfake-detection/.env")
HF_TOKEN = os.getenv("HF_TOKEN")

client = InferenceClient(api_key=HF_TOKEN, provider="auto")

client = InferenceClient(token=HF_TOKEN)

step 6 - Generate a natural-language explanation

provider="auto" — tells Hugging Face to route your request to whichever backend provider actually hosts the requested model, instead of assuming hf-inference (their own free serverless tier) has it

client.chat.completions.create(...) — this is the newer OpenAI-compatible method name (matches current HF docs); functionally the same as chat_completion, just the current recommended syntax

deepseek-ai/DeepSeek-V3-0324 — a model confirmed (per HF's own current documentation) to be actively served through their provider network

In [6]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto"  # automatically picks whichever provider actually hosts the model
)

def generate_explanation(result):
    prediction = result["prediction"]
    confidence = result["confidence"]
    fake_prob = result["fake_probability"]
    real_prob = result["real_probability"]
    
    prompt = f"""Detection result: {prediction.upper()}
Confidence: {confidence}%
Fake probability: {fake_prob}%
Real probability: {real_prob}%

Write a short, 2-3 sentence explanation for the user about this deepfake
detection result. Be honest this is a statistical prediction, not certainty."""

    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3-0324",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    
    return response.choices[0].message.content

try:
    explanation = generate_explanation(result)
    print(explanation)
except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR MESSAGE:", str(e))

ERROR TYPE: HfHubHTTPError
ERROR MESSAGE: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/v1/chat/completions' (Amz CF ID: oKJ1c5hvd8VUtBnvZScNwZ54E8xSGYuyv8OT9hPtMJTSwVDtwbwXiA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504


combine prediction + explanation into one clean function

In [7]:
def predict_with_explanation(image_path):
    result = predict_image(image_path)
    explanation = generate_explanation(result)
    result["explanation"] = explanation
    return result

# Test it
final_result = predict_with_explanation(test_image_path)
print(final_result)

{'prediction': 'real', 'confidence': 100.0, 'fake_probability': 0.0, 'real_probability': 100.0, 'explanation': 'This result indicates the detection system strongly believes the content is authentic, with 100% confidence in its prediction. However, no deepfake detector is infallible—this is a statistical assessment, not absolute proof. Always consider additional context or verification for high-stakes decisions.'}


Step 7 - Gradio interface to include the explanation

Gradio is a free, open-source Python library that turns a Python function into a shareable web interface — without you writing any HTML, CSS, or JavaScript.

Why it matters for your project specifically

Your predict_with_explanation() function is just Python code right now — great for you to test in a notebook, but not usable by anyone else without technical setup. Gradio wraps it in a proper web page: a file-upload button, a "submit" flow, and a display area for results — all auto-generated from a few lines of code.

How it works, conceptually

You give Gradio three things:

A function — your existing prediction logic (gradio_predict)
An input type — gr.Image(...) tells it "show an image upload widget"
An output type — gr.Textbox(...) tells it "display the function's return value as text"

Gradio then auto-builds a webpage matching those input/output types, wires up the "run the function when the user submits" logic, and serves it — either locally (http://127.0.0.1:7860, only visible on your own machine) or with a temporary public shareable link if you pass share=True to .launch().

In [8]:
import gradio as gr

def gradio_predict(image):
    temp_path = "D:/Deepfake-detection/outputs/temp_upload.jpg"
    image.save(temp_path)
    
    result = predict_with_explanation(temp_path)
    
    output_text = (
        f"Prediction: {result['prediction'].upper()}\n"
        f"Confidence: {result['confidence']}%\n\n"
        f"Fake probability: {result['fake_probability']}%\n"
        f"Real probability: {result['real_probability']}%\n\n"
        f"Explanation:\n{result['explanation']}"
    )
    return output_text

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=gr.Textbox(label="Detection Result", lines=10),
    title="AI-Powered Deepfake Detection System",
    description="Upload a face image to check if it's Real or AI-generated/Deepfake, with an AI-generated explanation."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
print("Token loaded:", HF_TOKEN is not None and len(HF_TOKEN) > 10)

Token loaded: True
